In [ ]:
import os
import json
from PIL import Image
import pandas as pd
import numpy as np
import random

random.seed(0)

# define your classes and folder names
classes = {'bike': 0, 'bus': 1, 'car': 2, 'person': 3, 'sign': 4, 'motor': 5, 'light': 6, 'truck': 7}
data_type = "rgb"
data_mode = "train"
folder_name = "images_{}_train".format(data_type)  # Adjust for val/test as needed

# Load the COCO format JSON
with open(f"/kaggle/input/teledyne-flir-adas-thermal-dataset-v2/FLIR_ADAS_v2/{folder_name}/coco.json", 'r') as f:
    coco = json.load(f)

# initialize a DataFrame to hold the image patch info
columns = ['patch_filename', 'original_filename', 'class', 'bbox', 'image_id']
patch_df = pd.DataFrame(columns=columns)

# Map filename and additional info
filename_id_map = {}
for image_info in coco['images']:
    try:
        hour = image_info['extra_info']['hours']
    except KeyError:
        hour = None
    filename_id_map[image_info['id']] = [image_info['file_name'], hour]

# Process each annotation to extract images
for annotation in coco['annotations']:
    image_id = annotation['image_id']
    filename, _ = filename_id_map[image_id]
    cls_name = coco['categories'][annotation['category_id']-1]['name']
    
    if cls_name in classes:
        img_path = os.path.join('/kaggle/input/teledyne-flir-adas-thermal-dataset-v2/FLIR_ADAS_v2', folder_name, filename)
        img = Image.open(img_path).convert('RGB')
        left, top, width, height = annotation['bbox']
        right, bottom = left + width, top + height
        
        # Check for large targets and expand bounding box by 25%
        if width * height * 1.25 >= 1000:
            cropped_img = img.crop((left - width // 8, top - height // 8, right + width // 8, bottom + height // 8))
            target_dir = f'./data_FLIR/{data_mode}_{data_type}_upsc/{cls_name}'
            os.makedirs(target_dir, exist_ok=True)
            patch_filename = f"{filename.split('/')[-1][:-4]}_{annotation['id']}.jpg"
            cropped_img.save(os.path.join(target_dir, patch_filename))
            
            # Add a new row to the DataFrame for this patch
            new_row = pd.DataFrame({
                'patch_filename': [patch_filename],
                'original_filename': [filename],
                'class': [cls_name],
                'bbox': [annotation['bbox']],
                'image_id': [image_id]
            })
            patch_df = pd.concat([patch_df, new_row], ignore_index=True)

patch_df.to_csv('./data_FLIR/patches_info.csv', index=False)